# 07 · RAG & 긴 문맥(Long-Context) 프롬프트 팁 검증

이 노트북은 긴 문맥(수천 토큰 규모의 입력)을 다룰 때 자주 나타나는 4가지 문제와 그 해법을, `gpt-5-nano`로 실제로 돌려보며 확인한다.
- RAG(Retrieval-Augmented Generation): 외부 문서를 프롬프트에 함께 넣어, 모델이 그 내용을 바탕으로 답하게 하는 방식.

| 팁 | 한 줄 요지 | 무엇을 측정하나 |
|----|-----------|-----------------|
| **2. 플레이북 압축** | 에이전트끼리 작업을 넘길 때 앞 단계의 긴 대화를 그대로 넘기면 입력 토큰이 계속 늘어난다. 중간에서 꼭 필요한 사실만 불릿으로 요약해 넘긴다. | 원문 그대로 전달 vs 요약 후 전달의 `prompt_tokens` 차이 |
| **12. 변수 위치 스와핑** | 같은 문서라도 프롬프트의 맨 위에 두느냐 맨 아래에 두느냐에 따라 모델이 정보를 회수하는 정도가 달라진다(Lost in the Middle). | 정답 단서 회수 성공/실패 비교 |
| **32. Needle in a Haystack 웜업** | 긴 채움 텍스트 안에 사실 하나(needle)를 심고, 프롬프트 끝에서 "먼저 그 사실부터 말하라"고 지시해 회수율을 높인다. Needle in a Haystack은 긴 문서 속에 숨긴 사실 하나를 찾게 하는 테스트를 말한다. | 웜업 지시 없음 vs 있음의 needle 회수율 |
| **38. 문서 간 오염 차단** | 두 문서를 동시에 주면 모델이 두 문서의 내용을 섞어서 답하기 쉽다. "각 문서는 독립적이며 서로 교차참조하지 말라"는 규칙을 넣는다. | 격리 규칙 없음 vs 있음의 정답 정확도 |

> 실행 환경 메모 (`gpt-5-nano` 실측 제약)
> - `temperature` 변경 불가(1 고정) → 창의성/보수성은 프롬프트 텍스트로만 통제
> - `max_tokens` 미지원 → `max_completion_tokens` 사용
> - 추론(reasoning) 모델이라 `max_completion_tokens`가 너무 작으면 추론 과정이 예산을 다 써서 본문 답변이 빈 문자열이 될 수 있음 → 넉넉히 주거나 `reasoning_effort="minimal"` 사용

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## Tip 2 · 플레이북 압축 (Playbook Compression)

**상황.** 여러 에이전트가 순서대로 작업을 넘기는 구조(예: 조사 에이전트 → 실행 에이전트)에서, 앞 단계가 만든 길고 장황한 이전 대화 전체를 다음 에이전트에게 그대로 넘기면, 단계를 넘길 때마다 입력 토큰이 쌓여 비용이 계속 커진다.

**해법.** 중간에 큐레이터 역할을 하나 두고, 긴 원문을 다음 단계가 실제로 필요로 하는 사실만 담은 불릿 요약(플레이북)으로 줄여서 넘긴다.

**검증 방법.**
- (A) 길게 늘어진 가짜 이전 대화 원문을 그대로 다음 프롬프트에 실어 보낸다.
- (B) 먼저 큐레이터에게 불릿 플레이북으로 요약시킨 뒤, 그 요약만 다음 프롬프트에 실어 보낸다.
- 두 경우의 다음 단계 `ask_meta(...)["prompt_tokens"]`를 비교해 입력 토큰이 얼마나 줄었는지 측정한다. (요약을 한 번 만드는 데 드는 비용은, 단계를 여러 번 넘길수록 나눠져 상쇄된다는 점도 함께 본다.)

In [ ]:
# [Tip 2] 원문 전달(A) vs 불릿 플레이북 압축 후 전달(B)의 입력 토큰 비교

# --- 길고 장황한 가짜 '이전 단계 대화 로그' (릴레이로 넘어온 원문) ---
long_transcript = (
    """[이전 조사 에이전트 로그 — 매우 장황함]
사용자: 우리 사내 개발 위키를 새로 정비하려고 하는데, 어디서부터 손대야 할지 모르겠어요.
에이전트: 네, 함께 정리해 보겠습니다. 우선 현황을 여쭤볼게요. 현재 위키는 어떤 도구로 운영되나요?
사용자: 예전엔 Confluence를 썼는데 라이선스 비용 때문에 작년에 사내 자체 MkDocs로 옮겼어요.
에이전트: 이관은 완료되었나요? 남아있는 Confluence 문서가 있는지 궁금합니다.
사용자: 절반쯤 옮겼고, 나머지 절반은 아직 Confluence에 남아 있어요. 특히 온보딩 문서랑 배포 런북이 안 옮겨졌어요.
에이전트: 알겠습니다. 그럼 검색은요? MkDocs 기본 검색을 쓰시나요?
사용자: 네 기본 검색인데, 한글 형태소 분석이 안 돼서 검색 품질이 나빠요. 다들 그냥 슬랙에 물어봐요.
에이전트: 접근 권한 관리는 어떻게 하고 계신가요?
사용자: 사실상 전사 공개예요. 인사·보안 문서까지 섞여 있어서 그게 좀 불안합니다.
에이전트: 문서 최신성은요? 오래된 문서가 방치되는 문제가 있나요?
사용자: 심각해요. 2021년 배포 절차 문서가 아직 상단에 떠서 신입이 그대로 따라 하다 사고가 난 적도 있어요.
에이전트: 정리하면 (1) Confluence 잔여 문서 이관, (2) 한글 검색 품질, (3) 권한 분리, (4) 오래된 문서 정리가 핵심 이슈로 보입니다. 우선순위는요?
사용자: 사고로 이어질 수 있는 4번(오래된 문서)이랑, 보안 리스크인 3번(권한)이 제일 급해요. 검색은 그 다음, 이관은 천천히요.
에이전트: 예산이나 마감 제약이 있을까요?
사용자: 이번 분기 안에 3번·4번은 끝내야 하고, 추가 예산은 거의 없어서 기존 MkDocs 안에서 해결해야 해요.
에이전트: 담당 인력은요?
사용자: 저 포함 프론트 2명이 남는 시간에 하는 거라, 자동화할 수 있는 건 최대한 자동화해야 합니다.
"""
)

# 다음 단계(실행 에이전트)가 받을 지시 템플릿 — 문서 자리에 A/B를 끼워 넣는다
next_step_instruction = (
    "다음은 이전 조사 단계에서 넘어온 맥락이다. 이를 근거로 "
    "'이번 분기 실행 계획'을 5개 항목 이내로 제안하라.\n\n[맥락]\n{ctx}"
)

# (A) 원문을 그대로 다음 단계에 전달 — 실제 호출은 하지 않고 입력 토큰만 측정
metaA = ask_meta(
    next_step_instruction.format(ctx=long_transcript),
    reasoning_effort="minimal",
    max_completion_tokens=400,
)

# (B) 큐레이터에게 먼저 불릿 플레이북으로 압축시킨다 (1회성 요약 비용)
curator = (
    "너는 릴레이 큐레이터다. 아래 대화 로그를 다음 실행 에이전트가 곧바로 행동에 쓸 수 있는 "
    "'불릿 플레이북'으로 압축하라. 잡담·인사·중복은 버리고, 확정된 사실/제약/우선순위만 "
    "6줄 이내 불릿으로. 각 불릿은 한 문장."
)
playbook = ask(curator, reasoning_effort="minimal", max_completion_tokens=500)
print("── 큐레이터가 만든 불릿 플레이북 ──")
print(playbook)

metaB = ask_meta(
    next_step_instruction.format(ctx=playbook),
    reasoning_effort="minimal",
    max_completion_tokens=400,
)

print("\n── 다음 단계 입력 토큰(prompt_tokens) 비교 ──")
print(f"(A) 원문 그대로 전달  : {metaA['prompt_tokens']:>5} tok")
print(f"(B) 플레이북 압축 전달: {metaB['prompt_tokens']:>5} tok")
saved = metaA['prompt_tokens'] - metaB['prompt_tokens']
ratio = saved / metaA['prompt_tokens'] * 100 if metaA['prompt_tokens'] else 0
print(f"→ 홉당 절감: {saved} tok ({ratio:.0f}%↓)  ※ 릴레이가 길수록 이 절감이 매 홉 누적된다")

## Tip 12 · 변수 위치 스와핑 (Lost in the Middle)

**상황.** 긴 문맥에서 모델은 맨 앞과 맨 뒤에 놓인 정보는 잘 찾아내지만, 가운데에 놓인 정보는 놓치는 경향이 있다. 이 현상을 Lost in the Middle(가운데 정보를 흘려버림)이라고 부른다. 그래서 같은 문서라도 프롬프트의 어느 위치에 두느냐에 따라 정답 회수율이 달라진다.

**해법.** 주입 문서를 프롬프트의 맨 위에 둘지 맨 아래에 둘지 바꿔가며 테스트하고, 정답 단서가 실제로 어디에 있는지에 맞춰 배치를 조정한다.

**검증 방법.**
- 긴 채움 문서를 만들고, 정답 단서("프로젝트 오리온의 승인 코드")를 문서 끝부분에 심는다.
- 질문("승인 코드가 뭐냐")은 고정하고, 문서를 (A) 질문 위 vs (B) 질문 아래에 배치한다.
- `keyword_hits`로 정답 코드가 회수됐는지 채점해 위치에 따른 효과를 관찰한다. (nano는 이 정도 문맥은 짧은 편이라 두 경우 모두 맞힐 수 있다. 차이가 재현되면 배치 규칙의 실전 근거로, 차이가 없으면 "이 규모에서는 위치 민감도가 낮다"는 관찰로 남긴다.)

In [ ]:
# [Tip 12] 같은 문서를 '질문 위'(A) vs '질문 아래'(B)에 두고 정답 회수 비교

# --- 채움 문단들: 그럴듯하지만 정답과 무관한 배경 소음 ---
filler = "\n".join(
    f"- 배경 메모 {i}: 분기 회의에서 물류·마케팅·인프라 관련 안건이 논의되었으나 최종 결정은 보류되었다."
    for i in range(1, 25)
)
# 정답 단서는 문서 '끝부분'에 심는다
doc = (
    "사내 기밀 브리핑 문서\n"
    + filler
    + "\n- 최종 승인 항목: 프로젝트 오리온의 배포 승인 코드는 'ORION-7788' 이며, 이 코드는 릴리스 담당자만 사용한다.\n"
    + "- 문서 끝."
)

question = "이 문서에서 '프로젝트 오리온'의 배포 승인 코드는 무엇인가? 코드만 정확히 답하라."

# (A) 문서를 질문 '위'에 배치
prompt_top = f"[문서]\n{doc}\n\n[질문]\n{question}"
# (B) 문서를 질문 '아래'에 배치
prompt_bottom = f"[질문]\n{question}\n\n[문서]\n{doc}"

ansA = ask(prompt_top, reasoning_effort="low", max_completion_tokens=200)
ansB = ask(prompt_bottom, reasoning_effort="low", max_completion_tokens=200)

compare("(A) 문서를 질문 '위'에 배치", ansA, "(B) 문서를 질문 '아래'에 배치", ansB)

gold = ["ORION-7788"]
print("(A) 회수 채점:", keyword_hits(ansA, gold))
print("(B) 회수 채점:", keyword_hits(ansB, gold))

## Tip 32 · Needle in a Haystack 웜업

**상황.** Needle in a Haystack은 긴 문서(수천 토큰) 안에 사실 하나(needle)를 심어두고 모델이 그것을 찾아내는지 보는 테스트다. 이때 모델이 그 사실을 찾지 못하거나 무시하는 경우가 있다. 특히 needle이 문서 앞이나 가운데에 있고 질문이 맨 끝에 있으면, 모델의 attention(어디에 집중해 읽는지)이 needle에서 멀어진다.

**해법.** 프롬프트 맨 끝의 지시문에서 "답하기 전에 먼저 비밀번호부터 그대로 말하라"처럼 needle을 명시적으로 먼저 꺼내게 만든다(웜업). 즉 회수를 먼저 시키고, 그다음에 본 과제를 하게 하는 순서다.

**검증 방법.**
- 반복되는 채움 문장 수천 토큰 사이 앞쪽에 needle("비밀번호는 블루-알파-42")을 심는다.
- (A) 웜업 지시 없이 곧바로 본 질문만 준다.
- (B) "먼저 비밀번호를 그대로 말한 뒤 본 과제를 수행하라"는 웜업 지시를 준다.
- `keyword_hits`로 needle 회수 성공 여부를 비교한다.

In [ ]:
# [Tip 32] needle 회수: 웜업 지시 없음(A) vs '먼저 비밀번호부터'(B)

# --- 건초더미 만들기: 반복 채움 문장 사이 '앞쪽'에 needle 삽입 ---
noise = "이 문단은 시스템 로그의 정상 하트비트 기록이며 특별한 의미가 없는 채움 텍스트다. "
haystack_parts = []
for i in range(1, 121):  # 채움 문장 120개 → 수천 토큰 규모
    haystack_parts.append(f"[{i:03d}] {noise}")
    if i == 8:  # needle을 앞쪽(8번째)에 몰래 심는다
        haystack_parts.append(
            "[NOTE] 참고: 이 시스템의 긴급 복구 비밀번호는 '블루-알파-42' 이다. 이 값은 한 번만 등장한다."
        )
haystack = "\n".join(haystack_parts)

task = "위 로그를 바탕으로 시스템이 정상 상태였는지 한 문장으로 판정하라."

# (A) 웜업 없음: 건초더미 + 본 과제만
prompt_no_warmup = f"{haystack}\n\n[과제]\n{task}"

# (B) 웜업: 끝 지시문에서 needle을 '먼저 그대로' 인출하도록 강제
prompt_warmup = (
    f"{haystack}\n\n[지시]\n"
    "답하기 전에, 위 로그에 등장한 '긴급 복구 비밀번호'를 먼저 그대로 한 줄로 적어라. "
    f"그다음 줄바꿈 후 아래 과제를 수행하라.\n[과제]\n{task}"
)

ansA = ask(prompt_no_warmup, reasoning_effort="low", max_completion_tokens=300)
ansB = ask(prompt_warmup, reasoning_effort="low", max_completion_tokens=300)

compare("(A) 웜업 지시 없음", ansA, "(B) '먼저 비밀번호부터' 웜업", ansB)

needle = ["블루-알파-42"]
print("(A) needle 회수:", keyword_hits(ansA, needle))
print("(B) needle 회수:", keyword_hits(ansB, needle))
print("→ (B)에서 비밀번호가 답 첫머리에 그대로 나오면 웜업이 attention을 끌어올린 것")

## Tip 38 · 문서 간 오염 차단 (Cross-Document Firewall)

**상황.** 서로 내용이 다른(상충하는) 두 문서(예: 회사A 환불 30일 / 회사B 환불 7일)를 동시에 주고 한쪽만 물어보면, 모델이 두 문서의 내용을 섞어서 틀린 답을 내놓기 쉽다.

**해법.** 시스템 규칙에 "각 문서는 서로 독립적이다. 질문에 명시된 문서만 사용하고, 다른 문서를 끌어와 섞지 마라"는 격리 규칙을 넣는다.

**검증 방법.**
- 환불 기간이 서로 다른 회사A(30일)·회사B(7일) 규정을 함께 준다.
- 질문은 "회사B의 환불 정책은?" (정답: 7일).
- (A) 격리 규칙 없이 vs (B) 격리 규칙 적용.
- `keyword_hits`로 정답("7일")이 들어갔는지와 오답("30일")이 빠졌는지를 채점해, 두 문서가 섞였는지 확인한다.

In [ ]:
# [Tip 38] 상충 문서 오염: 격리 규칙 없음(A) vs 방화벽 규칙(B)

# --- 상충하는 두 규정 문서 ---
doc_a = (
    "[문서: 회사A 이용약관]\n"
    "회사A는 구매일로부터 30일 이내 전액 환불을 보장한다. 환불은 영업일 기준 3일 내 처리된다. "
    "회사A의 고객센터 운영시간은 09:00~18:00 이다."
)
doc_b = (
    "[문서: 회사B 이용약관]\n"
    "회사B는 구매일로부터 7일 이내에만 환불이 가능하며, 개봉한 상품은 환불되지 않는다. "
    "회사B의 고객센터는 24시간 운영된다."
)
both_docs = doc_a + "\n\n" + doc_b
question = "회사B의 환불 가능 기간은 며칠인가? 숫자와 단위로 간단히 답하라."

# (A) 격리 규칙 없음
promptA = f"{both_docs}\n\n[질문]\n{question}"
ansA = ask(promptA, reasoning_effort="low", max_completion_tokens=200)

# (B) 방화벽(격리) 규칙을 시스템에 명시
firewall = (
    "너는 규정 조회 도우미다. 아래 규칙을 반드시 지켜라: "
    "(1) 제공된 문서들은 서로 완전히 독립적이다. "
    "(2) 질문에 명시된 회사의 문서만 근거로 사용하라. "
    "(3) 다른 회사 문서의 수치·조건을 교차참조하거나 혼합하지 마라. "
    "(4) 질문 대상 문서에 근거가 없으면 '해당 문서에 없음'이라고 답하라."
)
ansB = ask(promptA, system=firewall, reasoning_effort="low", max_completion_tokens=200)

compare("(A) 격리 규칙 없음", ansA, "(B) 방화벽(격리) 규칙 적용", ansB)

print("(A) 정답'7일' 포함:", keyword_hits(ansA, ["7"])["score"], "| 오염'30' 혼입:", keyword_hits(ansA, ["30"])["score"])
print("(B) 정답'7일' 포함:", keyword_hits(ansB, ["7"])["score"], "| 오염'30' 혼입:", keyword_hits(ansB, ["30"])["score"])
print("→ 정답은 '7'이 잡히고 '30'은 안 잡혀야 오염이 없는 것. (B)에서 더 깨끗하게 분리되는지 관찰")

## 노트북 요약 · 관찰 포인트

이 노트북은 긴 문맥을 다룰 때 그냥 전부 밀어 넣는 것이 아니라, 무엇을 / 어디에 / 어떻게 배치하느냐가 정확도와 비용을 좌우한다는 점을 보였다.

| 팁 | 무엇을 보면 검증됐다고 할 수 있나 |
|----|---------------------------------|
| **2. 플레이북 압축** | (B)의 `prompt_tokens`가 (A)보다 뚜렷이 작다. 단계를 여러 번 넘길수록 이 절감이 쌓여, 요약을 한 번 만드는 비용보다 커진다. |
| **12. 변수 위치 스와핑** | (A)/(B)의 `ORION-7788` 회수 결과 차이. 차이가 나면 Lost in the Middle이 실제로 나타난 것이고, 차이가 없으면 "이 규모에서는 위치 민감도가 낮다"는 관찰이다. 어느 쪽이든 정답 단서를 프롬프트 끝(질문 근처)에 두는 습관이 안전하다. |
| **32. Needle 웜업** | (B) 답 첫머리에 `블루-알파-42`가 그대로 먼저 나오고, needle 회수 score가 (A)보다 높다. 끝 지시문의 "먼저 X부터"가 attention을 끌어올린 결과다. |
| **38. 오염 차단** | (B)에서 정답 `7`은 잡히고 오답 `30`은 빠져서, 격리 규칙이 두 문서가 섞이는 것을 막는다. |

**공통 교훈.**
- 토큰은 공짜가 아니다 — 릴레이·RAG에서는 원문 대신 요약해서 넘겨라 (Tip 2).
- 모델의 attention은 위치에 따라 치우친다 — 핵심 단서는 가장자리(특히 끝)에 두고, 명시적으로 먼저 꺼내게 하라 (Tip 12·32).
- 여러 출처를 동시에 줄 때는 명시적인 격리 규칙으로 내용이 섞이는 것을 막아라 (Tip 38).

> 재현 메모: `gpt-5-nano`는 문맥 창이 넉넉하고 추론 모델이라, 짧은 실험에서는 (A)도 정답을 맞힐 수 있다. 그럴 때는 채움 텍스트 양(`range`)을 늘리거나 `reasoning_effort="minimal"`로 낮춰, 위치·웜업·격리의 효과가 잘 드러나는 구간을 찾아보라. 어느 쪽이든 실측값을 보고 판단하는 것이 이 노트북의 목적이다.